In [1]:
from unc_handling import UG_prompter
from DataLoader import DataLoader
from segmentation import Segmentation
from segmentation_util import combine_prompt_sets
from evaluation import Evaluator, compare_to_recontours, save_evaluation_results, evaluate_slice_by_slice
from uncertainty_util import remove_negative_prompts, remove_positive_prompts
from pathlib import Path
import numpy as np
import pandas as pd


root = r"C:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\data\LUNDPROBE\ExtendedSamples\development"

methods_available = ["raycast", "local_normals"]
propagation_styles = ['default', 'full', 'prompt_based', 'central_start', 'central_partitions']
method = methods_available[1]
propagation_style = propagation_styles[3]

rootpath = Path(root)
subjects = sorted([p.name for p in rootpath.iterdir() if p.is_dir()])
print(subjects)

data = DataLoader(parentfolder=root,subject_nr=3,volume_of_interest="CTVT",verbose=True)
unc_handler = UG_prompter(data=data)
seg_handler = Segmentation(data=data,use_mask_input_as_output_without_sam=False)

unc_handler.threshold_uncertainty_map(unc_threshold=None, target_mm=3.0, method="raycast", mode="median") #unc_threshold=0.033470
unc_handler.compute_band_thickness(method=method)
nietjes_prompts = unc_handler.generate_prompts_nietjes(unc_band_thr_mm=0.0,
        interpix_dist=5,
        pixel_interval=10,
        angle_step=5,
        method=method)


['newAcq_050f229dc2bdb64c', 'newAcq_0b4940fa31a1d650', 'newAcq_0cc559a8bd82a14a', 'newAcq_1b911d6cb2348f30', 'newAcq_1e0f8b9b01ce5f0b', 'newAcq_250d6075dd465a1a', 'newAcq_433a8d44fddd5b7f', 'newAcq_47ceabdbca398517', 'newAcq_486b7494ee9d71e7', 'newAcq_4a136e8fe320bd13']
Loaded subject newAcq_1b911d6cb2348f30 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024), Ground truth shape: (88, 1024, 1024)
Image spacing (z, y, x): (2.5, 0.46880000829696655, 0.46880000829696655)
Initilialized UG_prompter for subject 3 with volume of interest 'CTVT'
Image shape: (88, 1024, 1024), Mask shape: (88, 1024, 1024), Uncertainty map shape: (88, 1024, 1024).
Initialized Segmentation for subject 3 with volume of interest 'CTVT'
Mask shape: (88, 1024, 1024)
Building SAM predictor from checkpoint: checkpoints/MedSAM2_latest.pt


c:\Users\20202310\Desktop\MSc scriptie\MSc_Graduation_Project\sam2\modeling\sam\transformer.py:23: UserWarning: Flash Attention is disabled as it requires a GPU with Ampere (8.0) CUDA capability.
  OLD_GPU, USE_FLASH_ATTN, MATH_KERNEL_ON = get_sdpa_settings()


use_mask_input_as_output_without_sam: True
iter=00 | thr=0.199034 | band=0.59 mm | error=2.41
iter=01 | thr=0.099517 | band=1.05 mm | error=1.95
iter=02 | thr=0.049758 | band=1.41 mm | error=1.59
iter=03 | thr=0.024879 | band=1.88 mm | error=1.12
iter=04 | thr=0.012440 | band=2.34 mm | error=0.66
iter=05 | thr=0.006220 | band=2.81 mm | error=0.19
iter=06 | thr=0.003110 | band=3.52 mm | error=0.52
iter=07 | thr=0.004665 | band=3.16 mm | error=0.16
iter=08 | thr=0.005442 | band=2.99 mm | error=0.01
iter=09 | thr=0.005054 | band=3.05 mm | error=0.05
iter=10 | thr=0.005248 | band=3.05 mm | error=0.05
iter=11 | thr=0.005345 | band=2.99 mm | error=0.01
iter=12 | thr=0.005297 | band=3.05 mm | error=0.05
iter=13 | thr=0.005321 | band=3.05 mm | error=0.05
iter=14 | thr=0.005333 | band=2.99 mm | error=0.01
iter=15 | thr=0.005327 | band=3.05 mm | error=0.05
iter=16 | thr=0.005330 | band=2.99 mm | error=0.01
iter=17 | thr=0.005328 | band=3.05 mm | error=0.05
iter=18 | thr=0.005329 | band=3.05 mm |

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

from uncertainty_util import extract_bands, order_segmentation_pixels


# ============================================================
# DATA
# ============================================================

img = data.img
seg = data.mask.astype(bool)
unc_bin = unc_handler.unc_map_bin.astype(bool)

normals_by_slice = unc_handler.normals_by_slice


# ============================================================
# PRECOMPUTE ORDERED CONTOURS PER SLICE
# ============================================================

ordered_contours = {}

for z in range(seg.shape[0]):

    if not seg[z].any():
        continue

    seg_edge, _, _, _, _ = extract_bands(
        seg[z],
        unc_bin[z]
    )

    if not seg_edge.any():
        continue

    try:
        ordered_contours[z] = np.asarray(
            order_segmentation_pixels(seg_edge),
            dtype=float
        )
    except ValueError:
        continue


# ============================================================
# PLOT FUNCTION
# ============================================================

def plot_uncertainty_contour_normals(
    slice_idx,
    zoom_fraction=0.25,
    normal_length=15,
    normal_step=1,
    unc_alpha=0.35,
    contour_size=12,
    show=True,
):

    if slice_idx not in ordered_contours:
        print(f"No valid ordered contour for slice {slice_idx}.")
        return None

    if slice_idx not in normals_by_slice:
        print(f"No stored local normals for slice {slice_idx}.")
        return None

    image_slice = img[slice_idx]
    uncertainty_slice = unc_bin[slice_idx]

    contour = ordered_contours[slice_idx]

    normal_data = normals_by_slice[slice_idx]

    midpoints = np.asarray(
        normal_data["midpoints"],
        dtype=float
    )

    inner_normals = np.asarray(
        normal_data["inner_normals"],
        dtype=float
    )

    outer_normals = np.asarray(
        normal_data["outer_normals"],
        dtype=float
    )


    # --------------------------------------------------------
    # Subsample normals for readability
    # --------------------------------------------------------

    midpoints_sub = midpoints[::normal_step]
    inner_sub = inner_normals[::normal_step]
    outer_sub = outer_normals[::normal_step]


    # --------------------------------------------------------
    # Determine zoom region
    # --------------------------------------------------------

    all_points = np.vstack([
        contour,
        midpoints
    ])

    y_min, x_min = all_points.min(axis=0)
    y_max, x_max = all_points.max(axis=0)

    height = y_max - y_min
    width = x_max - x_min

    margin_y = max(10, height * zoom_fraction)
    margin_x = max(10, width * zoom_fraction)

    y0 = max(0, y_min - margin_y)
    y1 = min(image_slice.shape[0], y_max + margin_y)

    x0 = max(0, x_min - margin_x)
    x1 = min(image_slice.shape[1], x_max + margin_x)


    # --------------------------------------------------------
    # Plot
    # --------------------------------------------------------

    fig, ax = plt.subplots(figsize=(8, 8))


    # Base image
    ax.imshow(
        image_slice,
        cmap="gray",
        origin="upper",
    )


    # --------------------------------------------------------
    # Thresholded uncertainty overlay
    # --------------------------------------------------------

    uncertainty_overlay = np.ma.masked_where(
        ~uncertainty_slice,
        uncertainty_slice
    )

    ax.imshow(
        uncertainty_overlay,
        cmap="autumn",
        alpha=unc_alpha,
        origin="upper",
    )


    # --------------------------------------------------------
    # Ordered contour gradient
    # --------------------------------------------------------

    contour_order = np.arange(len(contour))

    scatter = ax.scatter(
        contour[:, 1],
        contour[:, 0],
        c=contour_order,
        cmap="turbo",
        s=contour_size,
        marker="s",                # SQUARE PIXELS
        linewidths=0,
        zorder=5,
    )


    # --------------------------------------------------------
    # Midpoints used for normal calculation
    # --------------------------------------------------------

    ax.scatter(
        midpoints_sub[:, 1],
        midpoints_sub[:, 0],
        s=20,
        facecolors="none",
        edgecolors="white",
        linewidths=0.8,
        zorder=7,
    )


    # --------------------------------------------------------
    # INNER NORMALS
    # --------------------------------------------------------

    ax.quiver(
        midpoints_sub[:, 1],
        midpoints_sub[:, 0],
        inner_sub[:, 1] * normal_length,
        inner_sub[:, 0] * normal_length,
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.003,
        zorder=8,
    )


    # --------------------------------------------------------
    # OUTER NORMALS
    # --------------------------------------------------------

    ax.quiver(
        midpoints_sub[:, 1],
        midpoints_sub[:, 0],
        outer_sub[:, 1] * normal_length,
        outer_sub[:, 0] * normal_length,
        angles="xy",
        scale_units="xy",
        scale=1,
        width=0.003,
        zorder=8,
    )


    # --------------------------------------------------------
    # Zoom
    # --------------------------------------------------------

    ax.set_xlim(x0, x1)
    ax.set_ylim(y1, y0)

    ax.set_aspect("equal")

    ax.set_title(
        f"Slice {slice_idx} | "
        f"Contour pixels: {len(contour)} | "
        f"Normals: {len(midpoints)}"
    )

    ax.set_axis_off()


    # --------------------------------------------------------
    # Contour-order colorbar
    # --------------------------------------------------------

    cbar = plt.colorbar(
        scatter,
        ax=ax,
        fraction=0.046,
        pad=0.04,
    )

    cbar.set_label("Ordered contour position")

    plt.tight_layout()

    if show:
        plt.show()

    return fig


# ============================================================
# ONLY ALLOW SLICES THAT ACTUALLY HAVE NORMALS
# ============================================================

available_slices = sorted(
    set(ordered_contours.keys())
    & set(normals_by_slice.keys())
)

if len(available_slices) == 0:
    raise ValueError(
        "No slices found containing both an ordered contour and stored normals."
    )


# ============================================================
# WIDGETS
# ============================================================

slice_dropdown = widgets.SelectionSlider(
    options=available_slices,
    value=available_slices[len(available_slices) // 2],
    description="Slice:",
    continuous_update=False,
)

zoom_slider = widgets.FloatSlider(
    value=0.25,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Margin:",
    continuous_update=False,
)

normal_length_slider = widgets.FloatSlider(
    value=15,
    min=2,
    max=50,
    step=1,
    description="Normal len:",
    continuous_update=False,
)

normal_step_slider = widgets.IntSlider(
    value=1,
    min=1,
    max=10,
    step=1,
    description="Normal step:",
    continuous_update=False,
)

alpha_slider = widgets.FloatSlider(
    value=0.35,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Unc alpha:",
    continuous_update=False,
)

contour_size_slider = widgets.IntSlider(
    value=12,
    min=2,
    max=40,
    step=2,
    description="Contour size:",
    continuous_update=False,
)


# ============================================================
# SAVE BUTTON
# ============================================================

save_button = widgets.Button(
    description="Save PNG",
    icon="save",
    button_style="success",
)

save_output = widgets.Output()


def save_current_plot(button):

    with save_output:

        clear_output(wait=True)

        fig = plot_uncertainty_contour_normals(
            slice_idx=slice_dropdown.value,
            zoom_fraction=zoom_slider.value,
            normal_length=normal_length_slider.value,
            normal_step=normal_step_slider.value,
            unc_alpha=alpha_slider.value,
            contour_size=contour_size_slider.value,
            show=False,               # don't display duplicate plot
        )

        filename = (
            f"uncertainty_normals_slice_"
            f"{slice_dropdown.value}.png"
        )

        fig.savefig(
            filename,
            dpi=300,
            bbox_inches="tight",
        )

        plt.close(fig)

        print(f"Saved: {filename}")


save_button.on_click(save_current_plot)


# ============================================================
# DISPLAY INTERACTIVE PLOT
# ============================================================

controls = widgets.VBox([
    slice_dropdown,
    zoom_slider,
    normal_length_slider,
    normal_step_slider,
    alpha_slider,
    contour_size_slider,
    save_button,
    save_output,
])


output = widgets.interactive_output(
    plot_uncertainty_contour_normals,
    {
        "slice_idx": slice_dropdown,
        "zoom_fraction": zoom_slider,
        "normal_length": normal_length_slider,
        "normal_step": normal_step_slider,
        "unc_alpha": alpha_slider,
        "contour_size": contour_size_slider,
    }
)


display(controls, output)

Output()